In [1]:
%reload_ext autoreload
%autoreload 2

import sys
import json
import pickle
from tqdm import tqdm

import numpy as np
import healpy as hp

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

from functools import partial


sys.path.append("..")

from models.np_model import NPModel
from models.scd import dnds
from likelihoods.npll_jax import log_like_np
from likelihoods.pll_jax import log_like_poisson
from einops import repeat

import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc_file("../../nptf-test/production/matplotlibrc")

/n/home07/yitians/.conda/envs/torch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# @partial(jax.jit, static_argnames=["m", "data"])
def ll(m, vd, data):
    # Get mixed pib template
    theta_pib = jnp.array(vd["theta_pib"])
    temp_pib = jnp.sum(theta_pib[:, None] * m.pib, 0)

    # Get mixed ics template
    theta_ics = jnp.array(vd["theta_ics"])
    temp_ics = jnp.sum(theta_ics[:, None] * m.ics, 0)

    S_gce = vd["S_gce"]

    temps = [m.temp_iso, m.temp_bub, m.temp_psc, temp_pib, temp_ics]
    temp_labels = ["iso", "bub", "psc", "pib", "ics"]
            
    mu = jnp.zeros_like(data)

    for temp, temp_label in zip(temps, temp_labels):
        
        if temp_label in ["pib", "ics"]:
            prior_lo, prior_hi = 1e-3, 14.
        else:
            prior_lo, prior_hi = 1e-3, 5.0
            
        S_temp = vd["S_{}".format(temp_label)]
        
        if temp_label in ["pib"]:
            
            temp_pib_mod = jnp.zeros_like(data)
            for ii in range(len(m.Ylm_temps)):
                Alm = vd["Alm_{}".format(ii)]
                temp_pib_mod += Alm * m.Ylm_temps[ii]
            
            temp_pib_mod = (1. + temp_pib_mod) * temp
            
            A_temp = S_temp / jnp.mean(temp_pib_mod[~m.normalization_mask])
            mu += A_temp * temp_pib_mod  
        else:
            A_temp = S_temp / jnp.mean(temp[~m.normalization_mask])
            mu += A_temp * temp     
                                        
    if m.vary_gamma:
        gamma_ps = vd["gamma_ps"]
        gamma_poiss = vd["gamma_poiss"]
    else:
        gamma_ps = 1.2 if m.non_poissonian else None
        gamma_poiss = 1.2

    temp_gce_nfw_ps = m.nfw_template.get_NFW2_template(gamma=gamma_ps) if m.non_poissonian else None
    temp_gce_nfw_poiss = m.nfw_template.get_NFW2_template(gamma=gamma_poiss)
        
    if m.non_poissonian:
        if m.vary_disk:
            zs = vd["zs"]
            C = vd["C"]
            temp_dsk = m.disk_template.get_template(zs=zs, C=C)
        else:
            temp_dsk = m.temp_dsk
            
    if m.bulge_hybrid:
        f_bulge_ps = vd["f_bulge_ps"]
        f_bulge_poiss = vd["f_bulge_poiss"]
        
        theta_bulge_poiss = jnp.array(vd["theta_bulge_poiss"])
        temp_bulge = jnp.sum(theta_bulge_poiss[:, None] * m.bulge_templates, 0)
    else:
        f_bulge_ps = 0. if m.non_poissonian else None
        f_bulge_poiss = vd["f_bulge_poiss"]
        temp_bulge = m.bulge_templates[0]

    # Normalize to same mean
    A_gce_nfw = S_gce / jnp.mean(temp_gce_nfw_poiss[~m.normalization_mask])
    A_gce_bulge = S_gce / jnp.mean(temp_bulge[~m.normalization_mask])
    temp_gce_poiss = (1 - f_bulge_poiss) * A_gce_nfw * temp_gce_nfw_poiss \
                        + f_bulge_poiss * A_gce_bulge * temp_bulge

    A_gce = S_gce / jnp.mean(temp_gce_poiss[~m.normalization_mask])
    mu += A_gce * temp_gce_poiss

    if m.non_poissonian:
        # Get mixed bulge template
        theta_bulge_ps = jnp.array(vd["theta_bulge_ps"])
        temp_bulge = jnp.sum(theta_bulge_ps[:, None] * m.bulge_templates, 0)

        # Normalize to same mean
        A_gce_nfw = 1 / jnp.mean(temp_gce_nfw_ps[~m.normalization_mask])
        A_gce_bulge = 1 / jnp.mean(temp_bulge[~m.normalization_mask])

        # Get hybrid template
        temp_gce_ps = (1 - f_bulge_ps) * A_gce_nfw * temp_gce_nfw_ps + f_bulge_ps * A_gce_bulge * temp_bulge

        npt_compressed = jnp.array([temp_gce_ps, temp_dsk])

        theta = []    

        for ips, ps in enumerate(["gce", "dsk"]):

            Sps = vd["Sps_{}".format(ps)]

            n1 = vd["n1_{}".format(ps)]
            n2 = vd["n2_{}".format(ps)]
            n3 = vd["n3_{}".format(ps)]
            sb1 = vd["sb1_{}".format(ps)]
            lambda_s = vd["lambdas_{}".format(ps)]

            theta_tmp = jnp.array([1., n1, n2, n3, sb1, lambda_s * sb1])

            s_ary = jnp.logspace(-1., 2., 1000)
            dnds_ary = dnds(s_ary, theta_tmp)

            A = Sps / jnp.mean(npt_compressed[ips][~m.normalization_mask] * jnp.trapz(s_ary * dnds_ary, s_ary))

            theta.append([A, n1, n2, n3, sb1, lambda_s * sb1])

        theta = jnp.array(theta)
            
    # Pad the last exposure region so that all are the same size
    exp_lens = [len(m.expreg_indices[i]) for i in range(len(m.expreg_indices))]
    n_pad = exp_lens[0] - exp_lens[-1]

    expreg_indices = jnp.zeros_like(m.expreg_indices)
    expreg_indices = expreg_indices.at[:-1].set(m.expreg_indices[:-1])
    expreg_indices = expreg_indices.at[-1].set(jnp.pad(m.expreg_indices[-1], (0, n_pad)))

    if m.non_poissonian:
        log_like_np_exp_vmapped = jax.vmap(log_like_np, in_axes=(0, 0, 1, 0, None, None, None, None))
    else:
        log_like_poisson_exp_vmapped = jax.vmap(log_like_poisson, in_axes=(0, 0))
            
    # Get relevant arrays for different exposure regions
    mu_batch = mu[~m.mask_roi][jnp.array(expreg_indices)]
    if m.non_poissonian:
        npt_compressed_batch = npt_compressed[:, ~m.mask_roi][:, jnp.array(expreg_indices)]
    data_batch = data[~m.mask_roi][jnp.array(expreg_indices)]

    exposure_multiplier = m.exposure_means_list / m.exposure_mean

    # Scale non-Poissonian parameters (norm divided by exposure ratio, breaks multiplied)
    if m.non_poissonian:
        theta = repeat(theta, "n_ps n_param -> n_exp n_ps n_param", n_exp=len(expreg_indices))
        theta = theta.at[:, :, 0].set(theta[:, :, 0] / exposure_multiplier[:, None])
        theta = theta.at[:, :, -1].set(theta[:, :, -1] * exposure_multiplier[:, None])
        theta = theta.at[:, :, -2].set(theta[:, :, -2] * exposure_multiplier[:, None])
        
    if m.non_poissonian:
        log_like_exp = log_like_np_exp_vmapped(theta, mu_batch, npt_compressed_batch, data_batch, m.f_ary, m.df_rho_ary, m.k_max, len(expreg_indices[0]))
    else:
        log_like_exp = log_like_poisson_exp_vmapped(mu_batch, data_batch)

    # Concatenate exposure regions
    loglike = jnp.concatenate(log_like_exp)[:len(mu[~m.mask_roi])]

    return loglike

In [7]:
class Args:
    pass
args = Args()
args.data = "base23fix_deltapsf_2"
args.i = 1
args.model = "base23fix_7exp_deltapsf"
args.n_exp = 7
args.truth_name = 'base230927'

# copied from ./1_fit.py
wdir = "/n/home07/yitians/fermi/fermi-prob-prog/production"
mask_roi = np.load(f"{wdir}/mask_roi.npy")
mask_norm = jnp.load(f"{wdir}/mask_norm.npy")

data = np.load(f"../outputs/sims/{args.data}.npy")[args.i]
if len(data) < hp.nside2npix(128):
    data_full = np.zeros(hp.nside2npix(128))
    data_full[~mask_norm] = data
    data_in = jnp.array(data_full, dtype=jnp.int32)
else:
    data_in = jnp.array(data, dtype=jnp.int32)

psf_tag = 'delta' if 'deltapsf' in args.model else 'king'
Model = NPModel
m = Model(data=data_in, psf_tag=psf_tag, n_exp=args.n_exp, custom_mask_roi=mask_roi)

vd = json.load(open(f"{wdir}/truth_dict_{args.truth_name}.json"))

Number of exposure regions: 7
Number of pixels in ROI: 6839
Using psf: delta
Max photon count is 122


In [8]:
def contract(s):
    s_new = s.copy()
    s_new['theta_pib'] = jnp.array([s['theta_pib_ModelO'], s['theta_pib_ModelA'], s['theta_pib_ModelF']]).T
    s_new['theta_ics'] = jnp.array([s['theta_ics_ModelO'], s['theta_ics_ModelA'], s['theta_ics_ModelF']]).T
    s_new['theta_bulge_poiss'] = jnp.array([s['theta_poiss_mcdermott2022'],
                                            s['theta_poiss_mcdermott2022_bbp'],
                                            s['theta_poiss_mcdermott2022_x'],
                                            s['theta_poiss_macias2019'],
                                            s['theta_poiss_coleman2019']]).T
    s_new['theta_bulge_ps'] = jnp.array([s['theta_ps_mcdermott2022'],
                                        s['theta_ps_mcdermott2022_bbp'],
                                        s['theta_ps_mcdermott2022_x'],
                                        s['theta_ps_macias2019'],
                                        s['theta_ps_coleman2019']]).T
    return s_new

def bestfit_dict(s):
    bf = {}
    for k, v in s.items():
        bf[k] = np.mean(v, axis=0)
    return bf

def get_i_dict(s, i):
    d = {}
    for k, v in s.items():
        d[k] = v[i]
    return d

In [9]:
samples_svi = pickle.load(open(f"../outputs/fit/svi_Dbase23fix_deltapsf_2_Mbase23fix_7exp_deltapsf_lr1e-3_sf/i0_n50000_ns5000.p", 'rb'))
# samples_svi1 = pickle.load(open(f"../outputs/fit/svi_Dbase23fix_Mbase23fix_7exp_iafm_np32/i0_n50000_ns5000.p", 'rb'))
# samples_svi2 = pickle.load(open(f"../outputs/fit/svi_Dbase23fix_Mbase23fix_7exp_iafm_np48/i0_n50000_ns5000.p", 'rb'))
samples_hmc = pickle.load(open(f"../outputs/fit/hmc_Dbase23fix_deltapsf_2_Mbase23fix_7exp_deltapsf/i0_n10000_ns0.p", 'rb'))

In [6]:
vd_map = pickle.load(open("tmp.p", 'rb'))

In [10]:
# best fit comparison
print('truth', ll(m, vd, data_in).sum())

vd_best_svi = bestfit_dict(contract(samples_svi))
print('svi', ll(m, vd_best_svi, data_in).sum())

# vd_best_svi1 = bestfit_dict(contract(samples_svi1))
# print('svi1', ll(m, vd_best_svi1, data_in).sum())

# vd_best_svi2 = bestfit_dict(contract(samples_svi2))
# print('svi2', ll(m, vd_best_svi2, data_in).sum())

vd_best_hmc = bestfit_dict(samples_hmc)
print('hmc', ll(m, vd_best_hmc, data_in).sum())

# print('map', ll(m, vd_map, data_in).sum())

truth -20366.040057632126
svi -20367.85032085645
hmc -20370.08522421796


In [44]:
# sample
ll_s = []
for i in tqdm(range(1000)):
    # vd = get_i_dict(contract(samples_svi), i)
    vd = get_i_dict(samples_hmc, i)
    ll_s.append(ll(m, vd, data_in).sum())

print(np.mean(ll_s), np.std(ll_s))

100%|██████████| 1000/1000 [01:15<00:00, 13.30it/s]

-19842.72458831901 2.7377526900899025
